In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
# return_X_y=True: trả thẳng về X và y, gọn hơn cách lấy data.data, data.target ở Buổi 1.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)   # (426, 30) (143, 30)

(426, 30) (143, 30)


In [3]:
# Ô 3 — Cách 1: tách thêm một tập validation
X_tr, X_val, y_tr, y_val = train_test_split(X_train,y_train, test_size=0.25, stratify=y_train, random_state=42)
print(X_tr.shape, X_val.shape, X_test.shape)

scores = {}

for d in range(1,9) :
  t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr,y_tr)
  scores[d] = round(t.score(X_val,y_val),3)
print(scores)

best_d = max(scores, key=scores.get)
# max(scores, key=scores.get): tìm key có value lớn nhất. key=scores.get bảo max so sánh theo điểm thay vì theo độ sâu. Khi hòa điểm (2, 6, 7 cùng 0,935), max lấy key gặp đầu tiên là 2.
print(best_d)

(319, 30) (107, 30) (143, 30)
{1: 0.869, 2: 0.935, 3: 0.925, 4: 0.916, 5: 0.907, 6: 0.935, 7: 0.935, 8: 0.888}
2


In [4]:
for rs in [0, 1, 2, 3, 4]:
    a, b, c, d_ = train_test_split(X_train, y_train, test_size=0.25,
                                   stratify=y_train, random_state=rs)
    # random_state từ 0 đến 4, tức 5 cách chia validation khác nhau.
    sc = [DecisionTreeClassifier(max_depth=d, random_state=42).fit(a, c).score(b, d_)
          for d in range(1, 9)]
    print(sc)
    print(rs, "độ sâu tốt nhất:", int(np.argmax(sc)) + 1, "điểm:", round(max(sc), 3))
    # np.argmax(sc): vị trí của điểm lớn nhất trong list (bắt đầu từ 0)

[0.9158878504672897, 0.9345794392523364, 0.9719626168224299, 0.9719626168224299, 0.9439252336448598, 0.9532710280373832, 0.9626168224299065, 0.9532710280373832]
0 độ sâu tốt nhất: 3 điểm: 0.972
[0.897196261682243, 0.9439252336448598, 0.9439252336448598, 0.9345794392523364, 0.9345794392523364, 0.9439252336448598, 0.9252336448598131, 0.9252336448598131]
1 độ sâu tốt nhất: 2 điểm: 0.944
[0.8785046728971962, 0.9532710280373832, 0.9345794392523364, 0.9345794392523364, 0.9439252336448598, 0.9439252336448598, 0.9439252336448598, 0.9439252336448598]
2 độ sâu tốt nhất: 2 điểm: 0.953
[0.8785046728971962, 0.9158878504672897, 0.897196261682243, 0.9065420560747663, 0.897196261682243, 0.9065420560747663, 0.9065420560747663, 0.9065420560747663]
3 độ sâu tốt nhất: 2 điểm: 0.916
[0.8691588785046729, 0.9065420560747663, 0.9065420560747663, 0.9158878504672897, 0.9252336448598131, 0.9252336448598131, 0.9252336448598131, 0.9252336448598131]
4 độ sâu tốt nhất: 5 điểm: 0.925


In [ ]:
# Ô 5 — Cách 2: cross-validation
# cv = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)
# for i,(tr_idx, va_idx) in enumerate(cv.split(X_train,y_train)) :
